# Temporal cross-validation design — train 2011–2020 only

**Zaakceptowane wejścia metodologiczne**

- supervised development sample: `N=23 218`;
- CV pool: train 2011–2020, `N=19 671`;
- external development validation: 2021–2022, `N=3 547`;
- preprocessing C: fold-only p1/p99 winsorization, median imputation,
  missing indicators i StandardScaler financial features;
- B bez indicators: obowiązkowa ablation;
- complete-case i no-winsorization: robustness only.

Frozen target, universe i raw `X_t` są wejściami tylko do odczytu.
Notebook nie trenuje modeli i nie używa lat 2023–2024. External
validation 2021–2022 jest wczytane wyłącznie do potwierdzenia stałej
liczebności; jego klasa ani cechy nie uczestniczą w projekcie foldów.

**Dodatkowa kontrola point-in-time.** Target dla feature year `t` staje
się znany dopiero z anchor filing `t+1`. Dlatego zwykłe
`train_end = validation_year - 1` nie wystarcza. Projekt stosuje:

1. jednoroczny feature-year embargo;
2. row-level cutoff: target training row musi być dostępny nie później
   niż najwcześniejszy `prediction_timestamp` w validation fold.


In [1]:
from pathlib import Path
import csv
import hashlib
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 72)

def project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs/x_t_pit_v1_freeze_manifest.yaml").is_file():
            return candidate
    raise FileNotFoundError("Nie znaleziono katalogu głównego projektu.")

ROOT = project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.modeling.preprocessing import (
    FEATURE_BLOCKS,
    FROZEN_BLOCK_COMPARISONS,
    FinancialPreprocessor,
    PreprocessingPolicy,
    features_for_blocks,
)
from src.modeling.temporal_cv import (
    MAIN_EXPANDING_WINDOW_FOLDS,
    fold_timeline,
    iter_point_in_time_folds,
    purge_overlapping_training_groups,
)

FEATURES = list(features_for_blocks(("L", "D", "R")))
ACCEPTED_X_STATUSES = {"available_core", "partially_available"}
PATHS = {
    "universe": ROOT / "data/processed/research_universe_pit.csv",
    "target": ROOT / "data/interim/target_candidate_v2_pit_b.csv",
    "x_t": ROOT / "data/processed/x_t_pit_v1_raw.csv",
    "target_application": ROOT / "data/processed/research_universe_pit_v1_1_0_target_pit_b_v1_0_0.csv",
}
EXPECTED = {
    "universe": ("a449c8145d1f46f954f12b1dfc079bb0b367c4f7f5edf3332a983ad7c1fb8182", 103099, 81),
    "target": ("473aa403dfd15822a15ce985f7698efe4a4e3a66bcf30b7634f0ca646805e0ff", 26917, 802),
    "x_t": ("0f1b35b9ffbb1fb1c1cdfb7dff12e3efd8fb38f60b33407ff2b2a8fb6b88397f", 64901, 1072),
    "target_application": ("ea42eb43018b2c8e238e2c4757260bb692e27edd5429628e28892f360f0f7f7d", 64901, 832),
}

def fingerprint_csv(path: Path) -> dict:
    digest = hashlib.sha256()
    byte_count = 0
    newline_count = 0
    with path.open("rb") as handle:
        while chunk := handle.read(8 * 1024 * 1024):
            digest.update(chunk)
            byte_count += len(chunk)
            newline_count += chunk.count(b"\n")
    with path.open(newline="", encoding="utf-8") as handle:
        columns = len(next(csv.reader(handle)))
    return {
        "sha256": digest.hexdigest(),
        "rows": max(newline_count - 1, 0),
        "columns": columns,
        "MiB": round(byte_count / 1024**2, 1),
    }

def read_development(path: Path, usecols: list[str]) -> pd.DataFrame:
    parts = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=10_000, low_memory=False):
        years = pd.to_numeric(chunk["feature_year"], errors="coerce")
        parts.append(chunk.loc[years.between(2011, 2022)].copy())
    return pd.concat(parts, ignore_index=True)

def build_fixed_sample() -> tuple[pd.DataFrame, pd.DataFrame]:
    x_columns = [
        "research_universe_company_year_id", "cik10", "feature_year", "split",
        "membership_status", "x_t_status", "economic_group_id",
        "prediction_timestamp",
    ] + [f"{feature}_value" for feature in FEATURES]
    target_columns = [
        "research_universe_company_year_id", "feature_year", "target_status",
        "target_candidate_v2_pit_b", "anchor_t1_accepted_at",
    ]
    x_frame = read_development(PATHS["x_t"], x_columns)
    target_frame = read_development(PATHS["target_application"], target_columns).rename(
        columns={
            "feature_year": "target_feature_year",
            "anchor_t1_accepted_at": "target_available_at",
        }
    )
    joined = x_frame.merge(
        target_frame,
        on="research_universe_company_year_id",
        how="left",
        validate="one_to_one",
    )
    fixed = joined.loc[
        joined["membership_status"].eq("eligible")
        & joined["target_status"].eq("available")
        & joined["x_t_status"].isin(ACCEPTED_X_STATUSES)
    ].copy()
    assert joined["feature_year"].eq(joined["target_feature_year"]).all()
    assert len(fixed) == 23_218
    assert fixed["split"].value_counts().to_dict() == {
        "train": 19_671, "validation": 3_547
    }
    assert fixed["research_universe_company_year_id"].is_unique
    assert fixed["economic_group_id"].notna().all()
    assert fixed["target_candidate_v2_pit_b"].isin([0.0, 1.0]).all()
    values = fixed[[f"{feature}_value" for feature in FEATURES]].rename(
        columns={f"{feature}_value": feature for feature in FEATURES}
    ).apply(pd.to_numeric, errors="raise").astype(float)
    values.index = fixed.index
    return fixed, values

def show(title: str, frame: pd.DataFrame | pd.Series) -> None:
    print(f"\n{title}")
    print("=" * len(title))
    print(frame.to_string())

fingerprints = {name: fingerprint_csv(path) for name, path in PATHS.items()}
for name, actual in fingerprints.items():
    expected_hash, expected_rows, expected_columns = EXPECTED[name]
    assert actual["sha256"] == expected_hash, f"Hash mismatch: {name}"
    assert actual["rows"] == expected_rows, f"Row mismatch: {name}"
    assert actual["columns"] == expected_columns, f"Column mismatch: {name}"

fixed_sample, feature_matrix = build_fixed_sample()
cv_pool = fixed_sample.loc[
    fixed_sample["split"].eq("train") & fixed_sample["feature_year"].between(2011, 2020)
].copy()
external_validation_index = fixed_sample.loc[
    fixed_sample["split"].eq("validation"),
    ["research_universe_company_year_id", "feature_year"],
].copy()
assert len(cv_pool) == 19_671
assert cv_pool["feature_year"].min() == 2011
assert cv_pool["feature_year"].max() == 2020
assert len(external_validation_index) == 3_547
assert external_validation_index["feature_year"].between(2021, 2022).all()

integrity = pd.DataFrame.from_dict(fingerprints, orient="index")
integrity["sha256_ok"] = True
integrity["shape_ok"] = True
show("Frozen input integrity", integrity[["rows", "columns", "MiB", "sha256_ok", "shape_ok"]])
print(
    f"\nFixed sample N={len(fixed_sample):,}; CV pool 2011–2020 N={len(cv_pool):,}; "
    f"external validation index 2021–2022 N={len(external_validation_index):,}"
)
print("No 2023–2024 rows are retained in memory.")



Frozen input integrity
                      rows  columns    MiB  sha256_ok  shape_ok
universe            103099       81  106.4       True      True
target               26917      802  169.3       True      True
x_t                  64901     1072  776.7       True      True
target_application   64901      832  383.7       True      True

Fixed sample N=23,218; CV pool 2011–2020 N=19,671; external validation index 2021–2022 N=3,547
No 2023–2024 rows are retained in memory.


## 1. Rozkład liczebności i klasy dodatniej w CV pool

Foldy są projektowane dopiero po sprawdzeniu rocznych liczebności.
`economic_group_id` służy wyłącznie jako identyfikator zależności — nie
jest i nie będzie predictorem.


In [2]:
yearly = cv_pool.groupby("feature_year", sort=True).agg(
    observations=("research_universe_company_year_id", "size"),
    negative_n=("target_candidate_v2_pit_b", lambda x: x.eq(0).sum()),
    positive_n=("target_candidate_v2_pit_b", lambda x: x.eq(1).sum()),
    positive_rate=("target_candidate_v2_pit_b", "mean"),
    economic_groups=("economic_group_id", "nunique"),
    ciks=("cik10", "nunique"),
)
yearly["positive_rate_pct"] = (100 * yearly.pop("positive_rate")).round(2)
show("Train 2011–2020 by feature year", yearly)



Train 2011–2020 by feature year
              observations  negative_n  positive_n  economic_groups  ciks  positive_rate_pct
feature_year                                                                                
2011                  1907        1550         357             1905  1907              18.72
2012                  2311        1943         368             2310  2311              15.92
2013                  2393        1969         424             2392  2393              17.72
2014                  2300        1812         488             2299  2300              21.22
2015                  2241        1735         506             2239  2241              22.58
2016                  2133        1782         351             2131  2133              16.46
2017                  1549        1292         257             1549  1549              16.59
2018                  1511        1184         327             1511  1511              21.64
2019                  1638        131

Każdy rok ma co najmniej 1 511 obserwacji i 219 klas dodatnich.
Początkowy point-in-time-safe train dla folda 2015 nadal ma ponad 6 tys.
obserwacji i ponad tysiąc pozytywów. Roczne validation windows dają więc
sześć punktów w czasie bez potrzeby łączenia lat.

## 2. Dlaczego potrzebny jest label-availability embargo

Dla validation feature year `v` porównujemy:

- naiwny train kończący się na `v-1`;
- feature-year embargo: train kończy się na `v-2`;
- główny PIT train: embargo plus usunięcie training rows, których
  `anchor_t1_accepted_at` jest późniejszy niż najwcześniejszy
  `prediction_timestamp` validation.

Ten cutoff jest liczony wyłącznie z timestampów dostępności, nie z
wartości cech ani targetu validation.


In [3]:
cv_pool_with_time = cv_pool.assign(
    prediction_at=pd.to_datetime(cv_pool["prediction_timestamp"], utc=True, errors="raise"),
    label_available_at=pd.to_datetime(cv_pool["target_available_at"], utc=True, errors="raise"),
)
availability_rows = []
for validation_year in range(2015, 2021):
    validation = cv_pool_with_time.loc[
        cv_pool_with_time["feature_year"].eq(validation_year)
    ]
    cutoff = validation["prediction_at"].min()
    for policy, train_end in (
        ("naive_no_gap", validation_year - 1),
        ("one_year_embargo_before_timestamp_cutoff", validation_year - 2),
    ):
        candidate = cv_pool_with_time.loc[
            cv_pool_with_time["feature_year"].between(2011, train_end)
        ]
        unavailable = candidate["label_available_at"].gt(cutoff)
        availability_rows.append(
            {
                "validation_year": validation_year,
                "candidate_policy": policy,
                "train_feature_years": f"2011–{train_end}",
                "validation_prediction_cutoff": cutoff.isoformat(),
                "candidate_train_n": len(candidate),
                "labels_unavailable_at_cutoff_n": int(unavailable.sum()),
                "labels_unavailable_pct": 100 * unavailable.mean(),
            }
        )
availability_audit = pd.DataFrame(availability_rows).set_index(
    ["validation_year", "candidate_policy"]
)
show("Label availability audit", availability_audit.round(2))



Label availability audit
                                                         train_feature_years validation_prediction_cutoff  candidate_train_n  labels_unavailable_at_cutoff_n  labels_unavailable_pct
validation_year candidate_policy                                                                                                                                                    
2015            naive_no_gap                                       2011–2014    2015-04-30T20:44:00+00:00               8911                            2440                   27.38
                one_year_embargo_before_timestamp_cutoff           2011–2013    2015-04-30T20:44:00+00:00               6611                             141                    2.13
2016            naive_no_gap                                       2011–2015    2016-04-28T22:09:00+00:00              11152                            2390                   21.43
                one_year_embargo_before_timestamp_cutoff           20

W naiwnym schemacie 9,5–27,4% kandydackiego train ma target jeszcze
niedostępny przy początku validation. Sam roczny embargo redukuje ten
problem do 0,3–2,1%, ale nie usuwa późnych filingów. Dlatego exact
timestamp cutoff jest częścią głównej definicji folda.

## 3. Główny expanding-window temporal CV

Dokładne foldy mają validation years 2015,…,2020. Symbol `TR` oznacza
training feature year, `EM` — jednoroczny label embargo, `VA` —
validation year. Po wyborze lat row-level timestamp cutoff wyklucza
nieliczne spóźnione training labels.


In [4]:
main_partitions = list(iter_point_in_time_folds(cv_pool))
main_rows = []
for fold, train, validation, audit in main_partitions:
    main_rows.append(
        {
            "fold": fold.name,
            "training_window": f"{fold.train_start}–{fold.train_end}",
            "embargo_years": ",".join(
                str(year) for year in range(fold.train_end + 1, fold.validation_start)
            ),
            "validation_window": f"{fold.validation_start}–{fold.validation_end}",
            "validation_prediction_cutoff": audit.validation_prediction_cutoff.isoformat(),
            "base_train_n": audit.base_train_rows,
            "late_labels_excluded_n": audit.label_unavailable_rows_excluded,
            "train_n": len(train),
            "train_positive_n": int(train["target_candidate_v2_pit_b"].sum()),
            "train_positive_rate_pct": 100 * train["target_candidate_v2_pit_b"].mean(),
            "validation_n": len(validation),
            "validation_positive_n": int(validation["target_candidate_v2_pit_b"].sum()),
            "validation_positive_rate_pct": 100 * validation["target_candidate_v2_pit_b"].mean(),
            "train_economic_groups": train["economic_group_id"].nunique(),
            "validation_economic_groups": validation["economic_group_id"].nunique(),
        }
    )
main_folds = pd.DataFrame(main_rows).set_index("fold")
timeline = fold_timeline()
show("Main PIT expanding-window folds", main_folds.round(2))
show("Timeline: TR=train, EM=embargo, VA=validation", timeline)



Main PIT expanding-window folds
          training_window embargo_years validation_window validation_prediction_cutoff  base_train_n  late_labels_excluded_n  train_n  train_positive_n  train_positive_rate_pct  validation_n  validation_positive_n  validation_positive_rate_pct  train_economic_groups  validation_economic_groups
fold                                                                                                                                                                                                                                                                                                  
fold_2015       2011–2013          2014         2015–2015    2015-04-30T20:44:00+00:00          6611                     141     6470              1112                    17.19          2241                    506                         22.58                   2955                        2239
fold_2016       2011–2014          2015         2016–2016    2016-04-28T22:09:00+0

### Preprocessing-within-CV

Dla każdego folda tworzony jest **nowy** `FinancialPreprocessor`.
`fit` widzi wyłącznie point-in-time-safe train fold. Validation jest
przekazywane wyłącznie do `transform`; hash parametrów przed i po
transformacji musi być identyczny. Ta sama procedura obowiązuje osobno
dla L, L+D i L+D+R oraz wariantu B.


In [5]:
PREPROCESSING_POLICY = PreprocessingPolicy(
    lower_quantile=0.01,
    upper_quantile=0.99,
    add_missing_indicators=True,
)

def parameter_hash(transformer: FinancialPreprocessor) -> str:
    payload = transformer.parameters_frame().to_csv(float_format="%.17g")
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]

preprocessing_rows = []
selected_parameter_rows = []
for fold, train, validation, _ in main_partitions:
    train_features = feature_matrix.loc[train.index, FEATURES]
    validation_features = feature_matrix.loc[validation.index, FEATURES]
    transformer = FinancialPreprocessor.for_blocks(
        ("L", "D", "R"), policy=PREPROCESSING_POLICY
    )
    transformer.fit(train_features)
    state_before = parameter_hash(transformer)
    transformed_validation = transformer.transform(validation_features)
    state_after = parameter_hash(transformer)
    assert state_before == state_after
    assert transformer.n_samples_seen_ == len(train)
    assert len(transformed_validation) == len(validation)
    assert transformed_validation.shape[1] == 34
    assert np.isfinite(transformed_validation.to_numpy()).all()
    preprocessing_rows.append(
        {
            "fold": fold.name,
            "preprocessor_fit_rows": transformer.n_samples_seen_,
            "validation_transform_rows": len(transformed_validation),
            "financial_columns": 17,
            "indicator_columns": 17,
            "output_columns": transformed_validation.shape[1],
            "parameters_hash_before_transform": state_before,
            "parameters_unchanged_after_validation_transform": state_before == state_after,
        }
    )
    for feature in ("roa_t", "revenue_growth_1y"):
        selected_parameter_rows.append(
            {
                "fold": fold.name,
                "feature": feature,
                "median": transformer.medians_[feature],
                "winsor_p1": transformer.lower_bounds_[feature],
                "winsor_p99": transformer.upper_bounds_[feature],
                "scaling_center": transformer.centers_[feature],
                "scaling_scale": transformer.scales_[feature],
            }
        )
preprocessing_audit = pd.DataFrame(preprocessing_rows).set_index("fold")
parameter_examples = pd.DataFrame(selected_parameter_rows).set_index(["fold", "feature"])

variant_scope = pd.DataFrame(
    [
        ("C main", "p1/p99 + median + indicators + StandardScaler", "fit fold train", "transform only"),
        ("B ablation", "p1/p99 + median + no indicators + StandardScaler", "fit fold train", "transform only"),
        ("complete-case robustness", "row filter inside each fold/block; no sample redefinition", "train/validation separately", "no learned stats"),
        ("no-winsor robustness", "median + indicators + StandardScaler", "fit fold train", "transform only"),
    ],
    columns=["variant", "operations", "training_partition", "validation_partition"],
).set_index("variant")
show("Fold-only preprocessing audit", preprocessing_audit)
show("Przykładowa ewolucja train-only parametrów", parameter_examples.round(5))
show("Preprocessing scope for main/ablation/robustness", variant_scope)



Fold-only preprocessing audit
           preprocessor_fit_rows  validation_transform_rows  financial_columns  indicator_columns  output_columns parameters_hash_before_transform  parameters_unchanged_after_validation_transform
fold                                                                                                                                                                                               
fold_2015                   6470                       2241                 17                 17              34                 52017894a51be6ba                                             True
fold_2016                   8761                       2133                 17                 17              34                 350cc915029eb8ff                                             True
fold_2017                  11089                       1549                 17                 17              34                 88e84d12286e6a7e                                       

## 4. `economic_group_id`: zależność i purged robustness CV

Wspólna grupa po obu stronach folda nie jest automatycznie leakage:
główny estimand obejmuje prognozowanie przyszłego company-year również
dla spółek/grup widzianych historycznie. Cechy pozostają point-in-time,
a `economic_group_id` nie trafia do model matrix.

Overlap oznacza jednak zależność obserwacji i może zawyżać precyzję
naiwnych row-level przedziałów ufności. Dlatego raportujemy koszt
robustness wariantu, który usuwa z train wszystkie grupy obecne w
validation. Validation oraz oś czasu pozostają bez zmian.


In [6]:
group_rows = []
for fold, train, validation, _ in main_partitions:
    purged_train, overlapping_groups = purge_overlapping_training_groups(
        train, validation
    )
    overlap = set(overlapping_groups)
    validation_seen = validation["economic_group_id"].astype(str).isin(overlap)
    group_rows.append(
        {
            "fold": fold.name,
            "main_train_n": len(train),
            "main_train_groups": train["economic_group_id"].nunique(),
            "validation_n": len(validation),
            "validation_groups": validation["economic_group_id"].nunique(),
            "overlap_groups": len(overlap),
            "validation_groups_seen_in_train_pct": 100 * len(overlap)
            / validation["economic_group_id"].nunique(),
            "validation_rows_seen_group_pct": 100 * validation_seen.mean(),
            "purged_train_n": len(purged_train),
            "purged_train_groups": purged_train["economic_group_id"].nunique(),
            "train_rows_removed_n": len(train) - len(purged_train),
            "train_rows_removed_pct": 100 * (1 - len(purged_train) / len(train)),
            "purged_train_positive_n": int(purged_train["target_candidate_v2_pit_b"].sum()),
            "purged_train_positive_rate_pct": 100
            * purged_train["target_candidate_v2_pit_b"].mean(),
        }
    )
group_audit = pd.DataFrame(group_rows).set_index("fold")

oof_year_rows = cv_pool.loc[cv_pool["feature_year"].between(2015, 2020)]
group_sizes = oof_year_rows.groupby("economic_group_id").size()
pooled_oof_group_structure = pd.Series(
    {
        "future_OOF_rows_2015_2020": len(oof_year_rows),
        "unique_economic_groups": group_sizes.size,
        "groups_with_multiple_OOF_rows": int(group_sizes.gt(1).sum()),
        "maximum_OOF_rows_per_group": int(group_sizes.max()),
    }
)
show("Economic-group overlap and purge cost", group_audit.round(2))
show("Structure relevant to later clustered OOF inference", pooled_oof_group_structure)



Economic-group overlap and purge cost
           main_train_n  main_train_groups  validation_n  validation_groups  overlap_groups  validation_groups_seen_in_train_pct  validation_rows_seen_group_pct  purged_train_n  purged_train_groups  train_rows_removed_n  train_rows_removed_pct  purged_train_positive_n  purged_train_positive_rate_pct
fold                                                                                                                                                                                                                                                                                                        
fold_2015          6470               2955          2241               2239            1771                                79.10                           79.12            2158                 1184                  4312                   66.65                      532                           24.65
fold_2016          8761               3255          2133  

Overlap dotyczy około 79–83% validation groups. Group purge usuwa około
41–67% point-in-time-safe train rows i podnosi udział klasy dodatniej w
pozostałym train. Nie jest więc neutralną korektą zależności — zmienia
estimand na generalizację do wcześniej niewidzianych grup.

**Decyzja:** bez purge jako główny temporal CV; dokładnie te same foldy z
group purge jako jeden robustness CV. Wyniki purged nie zastępują
głównych, lecz pokazują wrażliwość na company/group persistence.

### Późniejsze clustered bootstrap / inference

Po uzyskaniu out-of-fold predictions należy:

1. połączyć validation predictions z sześciu foldów, zachowując fold i
   `economic_group_id` wyłącznie jako metadata;
2. losować z powtórzeniami **economic groups**, nie pojedyncze wiersze;
3. w każdym wylosowanym klastrze zachować wszystkie jego company-years i
   statement scopes z odpowiednią multiplicity draw;
4. przeliczać metryki na każdym bootstrap replicate;
5. raportować clustered confidence intervals obok fold-level dispersion.

Ta sama zasada obowiązuje później dla 2021–2022 validation i finalnego
testu. Liczba replikacji, CI method i reguły dla rzadkich klas muszą być
zamrożone przed treningiem finalnych modeli.

## 5. Rola external validation 2021–2022

**Może być użyte:**

- jeden raz, po zakończeniu temporal CV i zamrożeniu całej kandydackiej
  pipeline;
- do zewnętrznej development oceny temporal transportability,
  stabilności metryk, calibration drift i ograniczeń;
- do decyzji go/no-go według wcześniej zapisanych kryteriów;
- do clustered inference po `economic_group_id`.

**Nie wolno na nim dostrajać:**

- feature blocks, sample, resolverów ani preprocessingu;
- percentyli winsoryzacji, median, scalerów lub indicators;
- model family, architektury, hiperparametrów, class weights/resampling;
- decision threshold, metryki wyboru, calibration method ani liczby
  iteracji/epok;
- wyboru między konfiguracjami po wielokrotnym oglądaniu validation.

Jeżeli locked pipeline nie spełni preregistered acceptance criteria,
można zadeklarować no-go albo utworzyć jawnie nową wersję metodologii;
nie wolno po cichu dostroić jej na 2021–2022 i nadal traktować tych lat
jako niezależnej validation.

**Freeze przed testem 2023–2024.** Przed jakimkolwiek otwarciem testu
muszą być zamrożone: sample policy, blocks, preprocessing/ablations,
CV, metric aggregation, model family i hyperparameters, seeds/training
budget, class-imbalance policy, threshold/calibration, refit policy oraz
clustered inference. Dopiero wtedy locked pipeline może zostać refit na
pełnym development 2011–2022 i jednokrotnie oceniona na test.

## 6. Rekomendacja końcowa

### Główny CV

- Sześć annual expanding-window foldów z validation 2015,…,2020.
- Training starts 2011, kończy się na `validation_year-2`.
- Rok `validation_year-1` jest label embargo.
- Dodatkowo train target must satisfy
  `target_available_at <= min(validation prediction_timestamp)`.
- Annual validation rows nigdy nie są w train tego folda.

### Dokładne foldy

| Fold | Train feature years | Embargo | Validation |
|---|---|---|---|
| fold_2015 | 2011–2013 | 2014 | 2015 |
| fold_2016 | 2011–2014 | 2015 | 2016 |
| fold_2017 | 2011–2015 | 2016 | 2017 |
| fold_2018 | 2011–2016 | 2017 | 2018 |
| fold_2019 | 2011–2017 | 2018 | 2019 |
| fold_2020 | 2011–2018 | 2019 | 2020 |

### Preprocessing-within-CV

Nowa instancja preprocessora per fold i block; `fit` tylko na
point-in-time-safe train, validation wyłącznie `transform`. Wariant B i
robustness checks dziedziczą identyczne temporal folds. Brak R nigdy nie
usuwa observation w głównym L+D+R.

### Economic groups

`economic_group_id` nigdy nie jest predictorem. Główny CV zachowuje
powtarzające się grupy, purged unseen-group CV jest robustness only, a
uncertainty jest później clusterowana po group ID.

### Decyzje wymagające zatwierdzenia

1. Sześć annual folds z one-year label embargo i exact timestamp cutoff.
2. Purged CV jako jedyny group-aware robustness, nie główny estimand.
3. Primary aggregation przyszłych CV metrics: rekomendowane raportowanie
   zarówno pooled OOF metric, jak i mean/dispersion across annual folds;
   jedna reguła rankingowa musi zostać zatwierdzona przed modelami.
4. Clustered bootstrap: liczba replikacji i CI method wymagają osobnej
   prerejestracji przed inference.
5. External validation 2021–2022 jako one-shot no-tune holdout.

**Nie wykonano treningu, model selection ani CV scoring. Test 2023–2024
nie został użyty.**
